In [271]:
from lark import Lark, ast_utils, Transformer, v_args
from typing import List
from dataclasses import dataclass
from lark.tree import Meta
import sys

In [272]:
this_module = sys.modules[__name__]

print(this_module)

<module '__main__'>


In [273]:
# source_code = '''
# let x = 5
# let y = 10

# def add(a, b):
#     return a + b
# end

# def func():
#     # TODO: Implement this function
# end


# let result = add(x, y)
# '''

In [274]:
source_code = """

let x = true

if x:
    display("x is true")
else:
    display("x is false")
end

"""

In [275]:
grammar = r''' 
    start: code_block

    code_block: statement*

    statement: let_statement
             | function_definition
             | return_statement
             | display_statement
             | if_statement

    let_statement: "let" NAME "=" expr

    function_definition: "def" NAME "(" [args] ")" ":" statement* "end"

    display_statement: "display" "(" expr ")"

    if_statement: "if" expr ":" statement* ("else" ":" statement*)? "end"

    args: expr ("," expr)*

    call: NAME "(" [args] ")"
    
 

    expr: DEC_NUMBER -> number
        | STRING -> string
        | "true" -> boolean_true
        | "false" -> boolean_false
        | call -> call
        | NAME -> name


    return_statement: "return" expr
    COMMENT: /#.*/
    NEWLINE: /(\r?\n)+/
    %ignore COMMENT
    %ignore NEWLINE
    %import python (NAME, DEC_NUMBER, STRING)
    %import common.WS
    %ignore WS

 
'''

In [276]:
def get_parser(grammar : str, parser_type: str = 'lalr', maybe_placeholders: bool = True):
    # Create and return a Lark parser instance using the specified grammar
    # Initialize the parser with the grammar and LALR parser algorithm
    return Lark(grammar=grammar, parser=parser_type, maybe_placeholders=maybe_placeholders)

In [277]:
# Test the parser with the source code
parsed_tree = get_parser(grammar).parse(source_code)
print(parsed_tree.pretty())

start
  code_block
    statement
      let_statement
        x
        boolean_true
    statement
      if_statement
        name	x
        statement
          display_statement
            string	"x is true"
        statement
          display_statement
            string	"x is false"



In [278]:
# Transform the parse tree into a more usable format (Abstract Syntax Tree)


class _AST(ast_utils.Ast):
    # Base class for all AST nodes
    pass 

class _Statement(_AST):
    # Base class for all statement nodes
    pass

class _Expression(_AST):
    # Base class for all expression nodes
    pass

# AST node for a block of code containing multiple statements
@dataclass
class CodeBlock(_AST, ast_utils.AsList):
    statements: List[_Statement]

# AST node for a variable name
@dataclass
class Name(_Expression):
    name: str 

# AST node for a literal number
@dataclass
class Number(_Expression, ast_utils.WithMeta):
    meta: Meta
    value: int

# AST node for a literal string
@dataclass
class String(_Expression, ast_utils.WithMeta):
    meta: Meta
    value: str

# AST Node for a let statement
@dataclass
class LetStatement(_Statement):

    name: str
    value: _Expression

# AST Node for a boolean literal
@dataclass
class Boolean(_Expression, ast_utils.WithMeta):
    meta: Meta
    value: bool


# AST Node for print statement
@dataclass
class DisplayStatement(_Statement):
    value: _Expression

@dataclass
class IfStatement(_Statement):

    condition: _Expression
    body: List[_Statement]
    orelse: List[_Statement] = None

@v_args(inline=True)
class ToAst(Transformer):
    
    def STRING(self, s):
    # Remove quotation marks
        return s[1:-1]

    def DEC_NUMBER(self, n):
        return int(n)
   
    def start(self, x):
        return x

    def boolean_true(self):
        return Boolean(meta=Meta(), value=True)

    def boolean_false(self):
        return Boolean(meta=Meta(), value=False)
    

    def statement(self, stmt):
        return stmt
    
    def if_statement(self, condition, body, orelse=None):
        return IfStatement(condition=condition, body=list(body), orelse=list(orelse) if orelse else None)


In [279]:
transformer = ast_utils.create_transformer(this_module, ToAst())

In [280]:
def parse(source_code, parser: Lark) -> _AST:
    tree = parser.parse(source_code)
    return transformer.transform(tree)

In [281]:
parser = get_parser(grammar)
ast = parse(source_code, parser)
ast.statements[1].body.value

String(meta=<lark.tree.Meta object at 0x000001EEDCE3E0F0>, value='x is true')

In [282]:
def transpile(node, indent=0):
    prefix = " " * indent

    if isinstance(node, CodeBlock):
        return "\n".join(
            transpile(statement, indent)
            for statement in node.statements
        )

    if isinstance(node, LetStatement):
        return f"{prefix}{str(node.name)} = {transpile(node.value)}"

    if isinstance(node, DisplayStatement):
        return f"{prefix}print({transpile(node.value)})"

    if isinstance(node, IfStatement):
        lines = [
            f"{prefix}if {transpile(node.condition)}:"
        ]
        if not hasattr(node.body, "__iter__"):
            lines.append(f"{transpile(node.body, indent + 4)}")
        else:
            lines.extend(
                transpile(statement, indent + 4)
                for statement in node.body 
            )

        if node.orelse:
            lines.append(f"{prefix}else:")
            if not hasattr(node.orelse, "__iter__"):
                lines.append(f"{transpile(node.orelse, indent + 4)}")
            else:
                lines.extend(
                    transpile(statement, indent + 4)
                    for statement in node.orelse
                )

        return "\n".join(lines)

    if isinstance(node, Number):
        return str(node.value)

    if isinstance(node, String):
        return repr(node.value)

    if isinstance(node, Name):
        return str(node.name)

    if isinstance(node, Boolean):
        return True if node.value else False

    raise TypeError(f"Unsupported AST node: {type(node).__name__}")

In [283]:
parser = get_parser(grammar)
ast = parse(source_code, parser)
python_code = transpile(ast)
print(python_code)

x = True
if x:
    print('x is true')
else:
    print('x is false')


In [285]:
# Execute python code as string
exec(python_code)

x is true


In [ ]:
source_code = """

let x = true

if x:
    display("x is true")
else:
    display("x is false")
end

"""

AttributeError: 'str' object has no attribute 'preview'

In [269]:
exec(source_code)

SyntaxError: invalid syntax (<string>, line 3)